# 03 · Identification, estimation, refutation and sensitivity (Sections 2.4, 2.5, 4.2)
For each of the 11 graph-compatible projects and each treatment (`ndev`, `entropy`, `sexp`, `rexp`):
1. verify the adjustment set is a valid, minimal back-door set in the graph;
2. estimate the effect with DoWhy's `backdoor.linear_regression`, with **no** automatic effect modifiers and the adjustment set fixed explicitly;
3. compute HC3 standard errors, CIs and p-values (Tables 3, 4, E1–E4);
4. run DoWhy's placebo, random-common-cause and data-subset refuters;
5. compute Cinelli–Hazlett omitted-variable sensitivity.

In [ ]:
# On Google Colab, uncomment the next line first:
# !pip install -q dowhy==0.14 statsmodels networkx
import warnings, numpy as np, pandas as pd, networkx as nx, statsmodels.api as sm
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

# Data source: Song & Minku (TSE 2022) replication package. Set BASE to a local folder to work offline.
BASE = "https://raw.githubusercontent.com/sunnysong14/ContinualPerformanceValidityTSE2022/main/data/"
Y, T = "days_to_first_fix", "author_date_unix_timestamp"

PROJECTS = {"brackets": "Brackets", "broadleaf": "Broadleaf", "camel": "Camel", "corefx": "CoreFX",
            "django": "Django", "fabric8": "Fabric8", "jgroup": "JGroup", "nova": "Nova", "rails": "Rails",
            "rust": "Rust", "tensorflow": "TensorFlow", "vscode": "VSCode", "wp-calypso": "WP-Calypso"}
# Projects on which the 33-edge graph is informative and not rejected (Section 4.1)
COMPATIBLE = ["brackets", "camel", "corefx", "django", "fabric8", "nova",
              "rails", "rust", "tensorflow", "vscode", "wp-calypso"]
METRICS = [T, "exp", "rexp", "fix", "ns", "nf", "sexp", "lt", "nuc", "ndev", "age", "la", "ld", "entropy", Y]


def load(name, horizon_q=0.9, return_info=False):
    """Defect-inducing commits of one project, with the right-censored tail removed.

    A commit made within `horizon` of the last commit in the data has had less time to be fixed than
    the project's 90th-percentile defect lifetime, so it is excluded (Section 2.2).
    """
    raw = pd.read_csv(f"{BASE}{name}.csv")
    raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")].rename(columns={"entrophy": "entropy"})
    bug = raw[raw.contains_bug == 1]
    horizon_days = bug[Y].quantile(horizon_q)
    kept = bug[bug[T] <= bug[T].max() - horizon_days * 86400]
    df = kept.replace([np.inf, -np.inf], np.nan).dropna(subset=METRICS).reset_index(drop=True)
    df["fix"] = df["fix"].astype(float)
    if return_info:
        return df, dict(project=PROJECTS[name], all_commits=len(raw), defect_inducing=len(bug),
                        horizon_days=round(horizon_days), after_censoring=len(kept), analysed=len(df))
    return df


In [ ]:
# Revised causal graph (Fig. 2): 14 nodes, 33 edges.
DAG33_EDGES = [
    (T, "exp"), (T, "rexp"), (T, "sexp"), (T, "nuc"), (T, "ndev"), (T, Y),
    ("exp", "rexp"), ("exp", Y),
    ("rexp", "nf"), ("rexp", Y),
    ("fix", "ns"), ("fix", "nf"), ("fix", Y),
    ("ns", "nf"), ("ns", "sexp"),
    ("nf", "lt"), ("nf", "nuc"), ("nf", "la"), ("nf", "entropy"), ("nf", Y),
    ("sexp", "exp"),
    ("lt", "nuc"), ("lt", "ndev"),
    ("nuc", "ndev"), ("nuc", "age"), ("nuc", "entropy"), ("nuc", Y),
    ("ndev", "age"), ("ndev", Y),
    ("age", Y),
    ("la", "entropy"), ("la", Y),
    ("entropy", Y),
]
# 27-edge variant without `fix` and `age` (same adjustment sets)
DROP = {"fix", "age"}
DAG27_EDGES = [(a, b) for a, b in DAG33_EDGES if a not in DROP and b not in DROP]

def to_graph(edges):
    g = nx.DiGraph(edges)
    assert nx.is_directed_acyclic_graph(g)
    return g

def to_dot(edges):
    return "digraph {\n" + "\n".join(f"    {a} -> {b};" for a, b in edges) + "\n}"

G33, G27 = to_graph(DAG33_EDGES), to_graph(DAG27_EDGES)
causal_graph = to_dot(DAG33_EDGES)          # DOT string accepted by DoWhy's CausalModel
print(f"G33: {G33.number_of_nodes()} nodes, {G33.number_of_edges()} edges | "
      f"G27: {G27.number_of_nodes()} nodes, {G27.number_of_edges()} edges")


G33: 14 nodes, 33 edges | G27: 12 nodes, 27 edges


## 1 · Adjustment sets

In [ ]:
import itertools

SETS = {"ndev": [T, "lt", "nuc"], "entropy": ["nf", "la", "nuc"], "sexp": [T, "ns"], "rexp": [T, "exp"]}

def is_backdoor_set(g, t, Z):
    if set(Z) & nx.descendants(g, t):
        return False
    h = g.copy(); h.remove_edges_from([(t, c) for c in list(g.successors(t))])
    return nx.is_d_separator(h, {t}, {Y}, set(Z))

def minimal_sets(g, t, max_size=4):
    cand = [v for v in g if v not in (t, Y) and v not in nx.descendants(g, t)]
    found = []
    for k in range(max_size + 1):
        for Z in itertools.combinations(cand, k):
            if is_backdoor_set(g, t, Z) and not any(set(f) <= set(Z) for f in found):
                found.append(Z)
    return found

for t, Z in SETS.items():
    assert is_backdoor_set(G33, t, Z) and is_backdoor_set(G27, t, Z)
    print(f"{t:8s} used {Z} | parents {sorted(G33.predecessors(t))} | all minimal sets {minimal_sets(G33, t)}")

ndev     used ['author_date_unix_timestamp', 'lt', 'nuc'] | parents ['author_date_unix_timestamp', 'lt', 'nuc'] | all minimal sets [('author_date_unix_timestamp', 'nuc', 'nf'), ('author_date_unix_timestamp', 'nuc', 'lt')]


entropy  used ['nf', 'la', 'nuc'] | parents ['la', 'nf', 'nuc'] | all minimal sets [('nuc', 'nf', 'la')]
sexp     used ['author_date_unix_timestamp', 'ns'] | parents ['author_date_unix_timestamp', 'ns'] | all minimal sets [('author_date_unix_timestamp', 'ns')]
rexp     used ['author_date_unix_timestamp', 'exp'] | parents ['author_date_unix_timestamp', 'exp'] | all minimal sets [('author_date_unix_timestamp', 'exp')]


## 2–3 · Estimates
DoWhy's point estimate is checked against an OLS fit on the same covariates; the OLS fit supplies HC3 standard errors.

Two DoWhy defaults are overridden:
* **Effect modifiers.** If none are given, DoWhy adds interactions between the treatment and the outcome's other parents. Passing `effect_modifiers=[]` to `CausalModel` is not enough in DoWhy 0.14 (an empty list is treated as unspecified), so it is passed to `estimate_effect`.
* **Adjustment set.** For `ndev` two minimal sets exist and DoWhy's choice between them is not stable across runs, so the set is fixed with `set_backdoor_variables`.

In [ ]:
from dowhy import CausalModel
import logging; logging.getLogger("dowhy").setLevel(logging.ERROR)

data = {p: load(p) for p in COMPATIBLE}
NODES = list(G33.nodes)

def dowhy_model(df, t):
    model = CausalModel(data=df[NODES], treatment=t, outcome=Y, graph=causal_graph)
    estimand = model.identify_effect(method_name="minimal-adjustment")
    estimand.set_backdoor_variables(SETS[t])
    estimate = model.estimate_effect(estimand, method_name="backdoor.linear_regression", effect_modifiers=[])
    return model, estimand, estimate

rows = []
for p in COMPATIBLE:
    df = data[p]
    for t, Z in SETS.items():
        _, _, est = dowhy_model(df, t)
        fit = sm.OLS(df[Y], sm.add_constant(df[[t] + Z])).fit(cov_type="HC3")
        b, se, sd = fit.params[t], fit.bse[t], df[t].std()
        assert np.isclose(b, est.value, rtol=1e-6), (p, t)
        rows.append(dict(project=p, treatment=t, n=len(df), beta=b, se=se, lo=b - 1.96 * se, hi=b + 1.96 * se,
                         p=fit.pvalues[t], sd=sd, per_sd=b * sd, per_sd_lo=(b - 1.96 * se) * sd,
                         per_sd_hi=(b + 1.96 * se) * sd))
E = pd.DataFrame(rows); E.to_csv("estimates11.csv", index=False)
print("saved estimates11.csv")

saved estimates11.csv


In [ ]:
fmt = lambda r, k: f"{r[k]:.4g}{'*' if r['p'] < 0.05 else ''}"
print("Table 3 – days per unit of treatment (* p < 0.05)")
display(E.assign(v=E.apply(lambda r: fmt(r, "beta"), axis=1)).pivot(index="project", columns="treatment", values="v")[list(SETS)])
print("Table 4 – days per 1 SD of treatment [95% CI]")
display(E.assign(v=E.apply(lambda r: f"{r.per_sd:+.2f} [{r.per_sd_lo:+.1f}, {r.per_sd_hi:+.1f}]", axis=1))
         .pivot(index="project", columns="treatment", values="v")[list(SETS)])

Table 3 – days per unit of treatment (* p < 0.05)


treatment,ndev,entropy,sexp,rexp
project,,,,
brackets,-0.06825,5.921,0.01156,-0.1384*
camel,-2.408*,-4.261,0.004439*,-0.02034*
corefx,-0.3121*,-10.43*,0.001589,0.2989
django,0.1738,-137.5*,0.009569*,0.01157
fabric8,1.556*,-10.08*,-0.01518*,0.04077
nova,0.5956*,-3.578,-0.005549,1.263e-05
rails,-0.8997*,-38.06*,0.01071,-0.002304
rust,-0.4271*,-1.702,-0.0003272,0.002849
tensorflow,-0.2494*,-14.43*,0.0002888*,-0.005872


Table 4 – days per 1 SD of treatment [95% CI]


treatment,ndev,entropy,sexp,rexp
project,,,,
brackets,"-1.63 [-36.0, +32.7]","+5.24 [-14.7, +25.2]","+5.75 [-6.2, +17.7]","-12.10 [-19.6, -4.6]"
camel,"-54.97 [-70.0, -40.0]","-5.91 [-22.1, +10.2]","+28.96 [+18.2, +39.7]","-8.45 [-12.9, -4.0]"
corefx,"-26.95 [-46.4, -7.5]","-21.80 [-43.6, -0.0]","+9.45 [-7.6, +26.5]","+19.49 [-19.0, +58.0]"
django,"+8.90 [-8.2, +26.0]","-143.29 [-155.6, -131.0]","+11.03 [+0.5, +21.6]","+4.52 [-7.7, +16.7]"
fabric8,"+25.12 [+9.0, +41.3]","-14.94 [-23.0, -6.8]","-23.60 [-29.6, -17.6]","+6.77 [-12.5, +26.1]"
nova,"+62.08 [+39.7, +84.5]","-3.93 [-10.6, +2.8]","-3.88 [-10.0, +2.3]","+0.03 [-3.1, +3.1]"
rails,"-126.34 [-164.2, -88.5]","-37.13 [-52.9, -21.4]","+8.55 [-2.6, +19.7]","-1.02 [-8.8, +6.8]"
rust,"-76.27 [-131.1, -21.4]","-2.64 [-22.4, +17.2]","-2.22 [-18.3, +13.9]","+0.95 [-8.0, +9.9]"
tensorflow,"-27.79 [-38.5, -17.1]","-18.00 [-22.0, -14.0]","+8.77 [+6.1, +11.4]","-1.15 [-4.2, +1.9]"


## 4–5 · Refutation tests and omitted-variable sensitivity
50 simulations per refuter (about 20–40 minutes for all 44 estimates). Lower `N_SIM` for a quick check.

In [ ]:
def partial_r2(t, dof):
    return t ** 2 / (t ** 2 + dof)

def sensitivity_ch(df, t, Z):
    """Cinelli & Hazlett (2020): robustness value and strongest observed benchmark."""
    fit = sm.OLS(df[Y], sm.add_constant(df[[t] + Z])).fit()
    dof = int(fit.df_resid); f = abs(fit.tvalues[t]) / np.sqrt(dof)
    rv = 0.5 * (np.sqrt(f ** 4 + 4 * f ** 2) - f ** 2)
    fa = max(f - stats.t.ppf(0.975, dof - 1) / np.sqrt(dof - 1), 0.0)
    rv_a = 0.5 * (np.sqrt(fa ** 4 + 4 * fa ** 2) - fa ** 2)
    bench = {}
    for c in Z:
        others = [a for a in Z if a != c]
        fd = sm.OLS(df[t], sm.add_constant(df[[c] + others])).fit()
        bench[c] = (round(partial_r2(fd.tvalues[c], int(fd.df_resid)), 4),
                    round(partial_r2(fit.tvalues[c], dof), 4))
    strongest = max(bench, key=lambda c: min(bench[c]))
    return dict(RV=round(rv, 4), RV_alpha=round(rv_a, 4), bench=strongest, bench_r2=bench[strongest],
                fragile=min(bench[strongest]) >= rv)

N_SIM = 50
rows = []
for p in COMPATIBLE:
    df = data[p]
    for t, Z in SETS.items():
        model, estimand, est = dowhy_model(df, t)
        np.random.seed(0)
        plc = model.refute_estimate(estimand, est, method_name="placebo_treatment_refuter",
                                    placebo_type="permute", num_simulations=N_SIM)
        rcc = model.refute_estimate(estimand, est, method_name="random_common_cause", num_simulations=N_SIM)
        dsr = model.refute_estimate(estimand, est, method_name="data_subset_refuter",
                                    subset_fraction=0.8, num_simulations=N_SIM)
        rows.append(dict(project=p, treatment=t, est=est.value,
                         placebo=plc.new_effect, placebo_p=plc.refutation_result["p_value"],
                         rcc=rcc.new_effect, rcc_p=rcc.refutation_result["p_value"],
                         dsr=dsr.new_effect, dsr_p=dsr.refutation_result["p_value"], **sensitivity_ch(df, t, Z)))
    print(PROJECTS[p], "done", flush=True)
R = pd.DataFrame(rows); R.to_csv("refutation11.csv", index=False)

Brackets done


Camel done


CoreFX done


Django done


Fabric8 done


...

Rails done


Rust done


TensorFlow done


VSCode done


WP-Calypso done


In [ ]:
M = R.merge(E[["project", "treatment", "p"]], on=["project", "treatment"]); sig = M[M.p < 0.05]
print(f"significant estimates: {len(sig)} of {len(M)}")
print(f"lowest refuter p-value: {M[['placebo_p', 'rcc_p', 'dsr_p']].min().min():.3f}")
print(f"placebo effect, max share of original: {(sig.placebo / sig.est).abs().max():.1%}")
print(f"random common cause, max change: {(sig.rcc / sig.est - 1).abs().max():.2%}")
print(f"data subset, max change: {(sig.dsr / sig.est - 1).abs().max():.1%}")
print(f"robustness value: {sig.RV.min():.1%} – {sig.RV.max():.1%} (median {sig.RV.median():.1%})")
print("fragile (strongest observed covariate would explain it away):",
      sig[sig.fragile][["project", "treatment"]].values.tolist())

significant estimates: 20 of 44
lowest refuter p-value: 0.313
placebo effect, max share of original: 11.9%
random common cause, max change: 0.32%
data subset, max change: 11.0%
robustness value: 1.8% – 18.8% (median 6.0%)
fragile (strongest observed covariate would explain it away): [['django', 'sexp']]
